In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import tensorflow as tf

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# read train csv file
df_train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
df_train = df_train.drop('Id', axis=1)
df_train.head()

## Data Exploration

In [ ]:
df_train.info()

In [ ]:
df_train.describe()

In [ ]:
plt.figure(figsize=(9, 8))
sns.distplot(df_train['SalePrice'], color='g', bins=100, hist_kws={'alpha': 0.4});

# the graph shows the SalePrice has right-skewed

In [ ]:
# check quantitative columns
list(set(df_train.dtypes.tolist()))

df_quant = df_train.select_dtypes(include=['int64', 'float64'])
print("Quantitative (Numerical) Columns:")
df_quant.head()

In [ ]:
df_quant.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8);

In [ ]:
# check qualitative columns
df_qual = df_train.select_dtypes(include=['object', 'category'])
print("\nQualitative (Categorical) Columns:")
df_qual.head()

## Data Preprocessing

In [ ]:
df_train.info()

null values: <br>
LotFrontage, Alley, MasVnrType, MasVnrArea, BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2, Electrical, FireplaceQu,GarageType, GarageYrBlt, GarageFinish, GarageQual, GarageCond, PoolQC, Fence, MiscFeature

In [ ]:
class HousingPreprocessor:
    '''
    Create a pipeline for data preprocessing which later can be used for test set
    '''
    def __init__(self):
        # Keep informative categorical columns; don't drop them
        self.drop_cols = []

        # Categorical columns where NA means absence of the feature
        self.none_cols = [
            "PoolQC", "MiscFeature", "Alley", "Fence", "MasVnrType",
            "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
            "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond"
        ]

        # Numeric cols where NA should be treated as 0 (absence)
        self.zero_cols = [
            "MasVnrArea",
            "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
            "BsmtFullBath", "BsmtHalfBath", "GarageYrBlt", "GarageCars", "GarageArea"
        ]

    def transform(self, df):
        df = df.drop(columns=[col for col in self.drop_cols if col in df.columns], errors='ignore')

        # Fill categorical NAs with explicit 'None' category
        for col in self.none_cols:
            if col in df.columns:
                df[col] = df[col].fillna("None")

        # Fill numeric NAs with 0 where appropriate
        for col in self.zero_cols:
            if col in df.columns:
                df[col] = df[col].fillna(0)

        # Neighborhood-median imputation for LotFrontage
        if "LotFrontage" in df.columns and "Neighborhood" in df.columns:
            df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))

        # Basic type consistency for categoricals
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].astype(str)

        # One-hot encode categoricals
        df = pd.get_dummies(df, drop_first=True)

        # Ensure bools become ints
        for col in df.select_dtypes(include='bool').columns:
            df[col] = df[col].astype('int32')

        return df

In [ ]:
preprocessor = HousingPreprocessor()
df_train = preprocessor.transform(df_train)

## Feature Engineering

In [ ]:
def feature_engineering(df):
    '''
    create a new engineer features from the existing columns
    to improve the model performance
    '''
    # Total bathrooms (full + 0.5*half) including basement
    if set(["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"]).issubset(df.columns):
        df["TotalBaths"] = (
            df["FullBath"].fillna(0)
            + 0.5 * df["HalfBath"].fillna(0)
            + df["BsmtFullBath"].fillna(0)
            + 0.5 * df["BsmtHalfBath"].fillna(0)
        )

    # Total square footage (basement + floors)
    if set(["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]).issubset(df.columns):
        df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"].fillna(0) + df["2ndFlrSF"].fillna(0)

    # Porches
    for col in ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    if set(["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]).issubset(df.columns):
        df["TotalPorchSF"] = df["OpenPorchSF"] + df["EnclosedPorch"] + df["3SsnPorch"] + df["ScreenPorch"]

    # Binary presence flags
    if "PoolArea" in df.columns:
        df["HasPool"] = (df["PoolArea"].fillna(0) > 0).astype("int32")
    if "GarageArea" in df.columns:
        df["HasGarage"] = (df["GarageArea"].fillna(0) > 0).astype("int32")
    if "Fireplaces" in df.columns:
        df["HasFireplace"] = (df["Fireplaces"].fillna(0) > 0).astype("int32")
    if "TotalPorchSF" in df.columns:
        df["HasPorch"] = (df["TotalPorchSF"].fillna(0) > 0).astype("int32")

    # Age-related features
    for col in ["YearBuilt", "YearRemodAdd", "YrSold"]:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    if set(["YrSold", "YearBuilt"]).issubset(df.columns):
        df["HouseAge"] = (df["YrSold"] - df["YearBuilt"]).clip(lower=0)
    if set(["YrSold", "YearRemodAdd"]).issubset(df.columns):
        df["RemodelAge"] = (df["YrSold"] - df["YearRemodAdd"]).clip(lower=0)
    if set(["YearBuilt", "YearRemodAdd"]).issubset(df.columns):
        df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype("int32")

    # Simple interaction: quality-weighted living area
    if set(["OverallQual", "GrLivArea"]).issubset(df.columns):
        df["QualXGrLivArea"] = df["OverallQual"].fillna(0) * df["GrLivArea"].fillna(0)

    return df

df_train = feature_engineering(df_train)
df_train.head()

## Data Splitting (Train and Validation)

In [ ]:
def split_dataset(dataset, test_ratio=0.30):
  test_indices = np.random.rand(len(dataset)) < test_ratio
  return dataset[~test_indices], dataset[test_indices]

train_ds_pd, valid_ds_pd = split_dataset(df_train)
print("{} examples in training, {} examples in validation.".format(
    len(train_ds_pd), len(valid_ds_pd)))

## Model Training

In [ ]:
label = 'SalePrice' # the target column

# Prepare features/target for train and validation sets
y_train = train_ds_pd[label]
X_train = train_ds_pd.drop(columns=[label])

y_valid = valid_ds_pd[label]
X_valid = valid_ds_pd.drop(columns=[label])

print(f"X_train shape: {X_train.shape} | X_valid shape: {X_valid.shape}")

In [ ]:
# No model listing needed for scikit-learn; this cell can be used to show feature count
X_train.shape, X_valid.shape

## 1. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    oob_score=True,
    bootstrap=True
)
rf

In [ ]:
rf.fit(X_train, y_train)

In [ ]:
# Out-of-bag score (R^2 on OOB samples)
print(f"OOB R2: {rf.oob_score_:.4f}")

In [ ]:
# Evaluation on validation split
pred_valid = rf.predict(X_valid)
mse = mean_squared_error(y_valid, pred_valid)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_valid, pred_valid)
r2 = r2_score(y_valid, pred_valid)
print(f"RMSE: {rmse:.4f}\nMAE: {mae:.4f}\nR2: {r2:.4f}")

In [ ]:
# Top 20 feature importances
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances.head(20)

In [ ]:
# Plot feature importances
plt.figure(figsize=(12, 6))
importances.head(20)[::-1].plot(kind='barh')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (RandomForestRegressor)')
plt.tight_layout()
plt.show()

## 2. XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=50
)

In [ ]:
# XGBoost evaluation
pred_xgb = xgb.predict(X_valid)
mse_xgb = mean_squared_error(y_valid, pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
mae_xgb = mean_absolute_error(y_valid, pred_xgb)
r2_xgb = r2_score(y_valid, pred_xgb)
print(f"XGBoost Results:")
print(f"RMSE: {rmse_xgb:.4f}\nMAE: {mae_xgb:.4f}\nR2: {r2_xgb:.4f}")

In [ ]:
# XGBoost feature importances
xgb_importances = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
xgb_importances.head(20)[::-1].plot(kind='barh')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

## 3. LightGBM

In [ ]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

lgbm = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=50)]
)

In [ ]:
# LightGBM evaluation
pred_lgbm = lgbm.predict(X_valid)
mse_lgbm = mean_squared_error(y_valid, pred_lgbm)
rmse_lgbm = np.sqrt(mse_lgbm)
mae_lgbm = mean_absolute_error(y_valid, pred_lgbm)
r2_lgbm = r2_score(y_valid, pred_lgbm)
print(f"LightGBM Results:")
print(f"RMSE: {rmse_lgbm:.4f}\nMAE: {mae_lgbm:.4f}\nR2: {r2_lgbm:.4f}")

## 4. Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=3,
    subsample=0.8,
    random_state=42
)

gbr.fit(X_train, y_train)

In [ ]:
# Gradient Boosting evaluation
pred_gbr = gbr.predict(X_valid)
mse_gbr = mean_squared_error(y_valid, pred_gbr)
rmse_gbr = np.sqrt(mse_gbr)
mae_gbr = mean_absolute_error(y_valid, pred_gbr)
r2_gbr = r2_score(y_valid, pred_gbr)
print(f"Gradient Boosting Results:")
print(f"RMSE: {rmse_gbr:.4f}\nMAE: {mae_gbr:.4f}\nR2: {r2_gbr:.4f}")

## Model Comparison

In [ ]:
# Compare all models
comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'LightGBM', 'Gradient Boosting'],
    'RMSE': [rmse, rmse_xgb, rmse_lgbm, rmse_gbr],
    'MAE': [mae, mae_xgb, mae_lgbm, mae_gbr],
    'R2': [r2, r2_xgb, r2_lgbm, r2_gbr]
})

comparison_df = comparison_df.sort_values('RMSE')
print("\n" + "="*60)
print("MODEL COMPARISON (sorted by RMSE - lower is better)")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

comparison_df.plot(x='Model', y='RMSE', kind='bar', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('RMSE (Lower is Better)')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=45)

comparison_df.plot(x='Model', y='MAE', kind='bar', ax=axes[1], legend=False, color='coral')
axes[1].set_title('MAE (Lower is Better)')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=45)

comparison_df.plot(x='Model', y='R2', kind='bar', ax=axes[2], legend=False, color='green')
axes[2].set_title('R² Score (Higher is Better)')
axes[2].set_ylabel('R²')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Stacking Ensemble

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

# Create stacking ensemble with the best performing models
estimators = [
    ('rf', RandomForestRegressor(n_estimators=400, max_depth=None, random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)),
    ('lgbm', LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1, verbose=-1)),
    ('gbr', GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42))
]

# Use Ridge regression as the final estimator (meta-learner)
stacking = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

print("Training stacking ensemble (this may take a few minutes)...")
stacking.fit(X_train, y_train)

In [ ]:
# Stacking Ensemble evaluation
pred_stack = stacking.predict(X_valid)
mse_stack = mean_squared_error(y_valid, pred_stack)
rmse_stack = np.sqrt(mse_stack)
mae_stack = mean_absolute_error(y_valid, pred_stack)
r2_stack = r2_score(y_valid, pred_stack)
print(f"\nStacking Ensemble Results:")
print(f"RMSE: {rmse_stack:.4f}\nMAE: {mae_stack:.4f}\nR2: {r2_stack:.4f}")

# Final comparison with stacking
final_comparison = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'LightGBM', 'Gradient Boosting', 'Stacking Ensemble'],
    'RMSE': [rmse, rmse_xgb, rmse_lgbm, rmse_gbr, rmse_stack],
    'MAE': [mae, mae_xgb, mae_lgbm, mae_gbr, mae_stack],
    'R2': [r2, r2_xgb, r2_lgbm, r2_gbr, r2_stack]
})

final_comparison = final_comparison.sort_values('RMSE')
print("\n" + "="*70)
print("FINAL MODEL COMPARISON (including Stacking Ensemble)")
print("="*70)
print(final_comparison.to_string(index=False))
print("="*70)
print(f"\nBest Model: {final_comparison.iloc[0]['Model']}")
print(f"Best RMSE: {final_comparison.iloc[0]['RMSE']:.4f}")

## Submission - Best Model (XGBoost)

In [ ]:
# Prepare test set and generate predictions using XGBoost (best model)
df_test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
ids = df_test.pop('Id')

# Apply same preprocessing pipeline
df_test_proc = preprocessor.transform(df_test)
df_test_proc = feature_engineering(df_test_proc)

# Align columns to training features (handle missing/extra columns)
missing_cols = [c for c in X_train.columns if c not in df_test_proc.columns]
for c in missing_cols:
    df_test_proc[c] = 0
    
extra_cols = [c for c in df_test_proc.columns if c not in X_train.columns]
df_test_proc = df_test_proc.drop(columns=extra_cols)

# Ensure same column order as training
df_test_proc = df_test_proc[X_train.columns]

print(f"Test set shape: {df_test_proc.shape}")
print(f"Training set shape: {X_train.shape}")
print(f"Columns match: {list(df_test_proc.columns) == list(X_train.columns)}")

# Generate predictions with XGBoost (best model)
preds_stacking = stacking.predict(df_test_proc)

# Create submission DataFrame
submission = pd.DataFrame({
    'Id': ids,
    'SalePrice': preds_xgb
})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print(f"  Total predictions: {len(submission)}")
print(f"  Price range: ${submission['SalePrice'].min():.2f} - ${submission['SalePrice'].max():.2f}")
print(f"  Mean price: ${submission['SalePrice'].mean():.2f}")

submission.head(10)